# Example: Skip-Gram with Full Softmax and Negative Sampling
In this example, we train a Skip-Gram model on the same toy corpus used in L9a. We first train with the full softmax objective, then implement Negative Sampling and compare the training dynamics. Finally, we compare the learned Skip-Gram embeddings to the Continuous Bag of Words (CBOW) embeddings from L9a.

> __Learning Objectives:__
> 
> By the end of this example, you should be able to:
>
> * __Skip-Gram training pairs__: Generate (target, context) training pairs from a corpus where each target word produces multiple context predictions. Contrast this with CBOW, where multiple context words predict one target.
> * __Full softmax vs Negative Sampling__: Train a Skip-Gram model using both the full softmax objective and Negative Sampling. Compare the loss curves and training time of each approach.
> * __Embedding comparison__: Extract word embeddings from the trained Skip-Gram model and compare cosine similarities to CBOW embeddings. Evaluate whether Skip-Gram captures different word relationships than CBOW.

Let's get started!
___

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading any needed resources, such as sample datasets, and setting up any required constants.

> The `Include.jl` file also loads external packages, various functions that we will use in the exercise, and custom types to model the components of our problem. It checks for a `Manifest.toml` file; if it finds one, packages are loaded. Other packages are downloaded and then loaded.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

### Data
We use the same toy corpus from the L9a examples. Let's build the `sentences::Array{String, 1}` variable containing our corpus.

In [ ]:
sentences = let
    sentences_array = Array{String,1}();
    push!(sentences_array, "I love machine learning and data science .");
    push!(sentences_array, "Machine learning is fun .");
    push!(sentences_array, "Machine learning is great .");
    push!(sentences_array, "I love coding machine learning in Julia !");
    push!(sentences_array, "Julia is a great programming language ?");
    push!(sentences_array, "I enjoy learning new things about data science , machine learning , and artificial intelligence .");
    sentences_array
end

Next, we build the vocabulary in the `vocabulary::Dict{String, Int64}` variable, where keys are unique words and values are their indices, and the inverse vocabulary in the `inverse_vocabulary::Dict{Int64, String}` variable.

In [ ]:
vocabulary, inverse_vocabulary = let
    vocabulary = Dict{String, Int64}();
    inverse_vocabulary = Dict{Int64, String}();
    index = 1;
    control_tokens = ["<bos>", "<eos>", "<pad>", "<unk>"];
    words = Set{String}();
    for sentence in sentences
        tmp = split(lowercase(sentence));
        push!(words, tmp...);
    end
    words_array = collect(words) |> sort;
    words_array = vcat(words_array, control_tokens);
    for word in words_array
        vocabulary[word] = index;
        inverse_vocabulary[index] = word;
        index += 1;
    end
    (vocabulary, inverse_vocabulary)
end

___

## Task 1: Generate Skip-Gram Training Pairs
Skip-Gram reverses the CBOW input/output: instead of (context, target), we generate (target, context_word) pairs. For each target word in a sentence, we produce one training pair per context word within the window.

> __Skip-Gram vs CBOW pairs__
>
> For a window size $m = 2$ and target word at position $t$, CBOW produces one pair: (sum of context one-hots, target one-hot). Skip-Gram produces $2m$ pairs: (target one-hot, context word 1), (target one-hot, context word 2), etc. This is why Skip-Gram generates more training signal per word occurrence.

Let's generate the pairs and store them in the `skipgram_pairs::Array{Tuple{Vector{Float64}, Vector{Float64}}, 1}` variable, where each element is a (target one-hot, context one-hot) tuple.

In [ ]:
skipgram_pairs = let

    # initialize -
    window_size = 2;
    vocab_size = length(vocabulary);
    pairs = Array{Tuple{Vector{Float64}, Vector{Float64}}, 1}(); # (target_onehot, context_onehot)

    # generate training pairs from each sentence -
    for sentence in sentences
        augmented_sentence = "<bos> " * sentence * " <eos>";
        words = split(lowercase(augmented_sentence)) .|> String;
        word_indices = [get(vocabulary, word, vocabulary["<unk>"]) for word in words];

        for i ∈ eachindex(word_indices)
            target_index = word_indices[i];
            target_onehot = zeros(Float64, vocab_size);
            target_onehot[target_index] = 1.0;

            # generate one pair per context word -
            left = max(1, i - window_size);
            right = min(length(word_indices), i + window_size);
            for j ∈ left:right
                if j == i
                    continue
                end
                context_onehot = zeros(Float64, vocab_size);
                context_onehot[word_indices[j]] = 1.0;
                push!(pairs, (target_onehot, context_onehot));
            end
        end
    end

    pairs
end;
println("Generated $(length(skipgram_pairs)) Skip-Gram pairs (vs CBOW: one pair per target position)")

___

## Task 2: Train Skip-Gram with Full Softmax
We train the Skip-Gram model using the full softmax objective. The forward pass is: $\mathbf{h} = \mathbf{W}_1 \mathbf{x}_{\text{target}}$, $\mathbf{u} = \mathbf{W}_2 \mathbf{h}$, $\hat{\mathbf{y}} = \text{softmax}(\mathbf{u})$. The loss is the cross-entropy between $\hat{\mathbf{y}}$ and the context word one-hot vector.

> __Full softmax training__
>
> Each (target, context) pair is one training example. The gradient of cross-entropy through softmax is $\hat{\mathbf{y}} - \mathbf{y}_c$, the same form as in CBOW. The difference is that Skip-Gram processes $2m$ pairs per target word position, while CBOW processes one.

We store the trained weight matrices in `W₁_sg::Matrix{Float64}` and `W₂_sg::Matrix{Float64}`, and the per-epoch average loss in `loss_history_softmax::Vector{Float64}`.

In [ ]:
W₁_sg, W₂_sg, loss_history_softmax = let

    # hyperparameters -
    vocab_size = length(vocabulary);
    d_h = 5;           # embedding dimension (same as L9a CBOW)
    η = 0.01;          # learning rate
    num_epochs = 500;  # number of training epochs

    # initialize weight matrices -
    W₁ = randn(d_h, vocab_size) * 0.1;
    W₂ = randn(vocab_size, d_h) * 0.1;

    # training loop -
    loss_history = zeros(num_epochs);
    for epoch ∈ 1:num_epochs
        epoch_loss = 0.0;
        for (x_target, y_context) in skipgram_pairs

            # forward pass -
            h = W₁ * x_target;                     # hidden layer (selects column of W₁)
            u = W₂ * h;                             # output logits
            ŷ = softmax(u);                         # predicted probabilities

            # cross-entropy loss -
            loss = -sum(y_context .* log.(ŷ .+ 1e-12));
            epoch_loss += loss;

            # backward pass -
            δ_u = ŷ - y_context;                    # ∂L/∂u
            ∂L_∂W₂ = δ_u * h';                     # ∂L/∂W₂
            δ_h = W₂' * δ_u;                       # backprop to hidden
            ∂L_∂W₁ = δ_h * x_target';              # ∂L/∂W₁

            # gradient descent update -
            W₁ .-= η * ∂L_∂W₁;
            W₂ .-= η * ∂L_∂W₂;
        end
        loss_history[epoch] = epoch_loss / length(skipgram_pairs);
    end

    (W₁, W₂, loss_history)
end;
println("Full softmax training complete. Final loss: $(round(loss_history_softmax[end], digits=4))")

___

## Task 3: Train Skip-Gram with Negative Sampling
Instead of computing the full softmax over the vocabulary, Negative Sampling uses a binary classification objective. For each (target, context) pair, we compute sigmoid scores for the positive context word and $k$ randomly sampled negative words.

> __Negative Sampling objective__
>
> For target embedding $\mathbf{h}$ and context word embedding $\mathbf{w}_c^{(2)}$ (row of $\mathbf{W}_2$), the loss is: $\mathcal{L} = -\log\sigma(\mathbf{w}_c^{(2)\top}\mathbf{h}) - \sum_{i=1}^{k}\log\sigma(-\mathbf{w}_{n_i}^{(2)\top}\mathbf{h})$. Negative words are sampled from a noise distribution proportional to word frequency raised to the $3/4$ power.

First, we build the noise distribution in the `noise_distribution::Vector{Float64}` variable, where each entry is the sampling probability for the corresponding word in the vocabulary.

In [ ]:
# build the noise distribution P_n(w) ∝ f(w)^(3/4)
noise_distribution = let
    vocab_size = length(vocabulary);
    word_counts = zeros(Float64, vocab_size);
    for sentence in sentences
        augmented_sentence = "<bos> " * sentence * " <eos>";
        words = split(lowercase(augmented_sentence)) .|> String;
        for word in words
            idx = get(vocabulary, word, vocabulary["<unk>"]);
            word_counts[idx] += 1.0;
        end
    end
    freq_34 = word_counts .^ 0.75;
    freq_34 / sum(freq_34)
end;

Now we train the Skip-Gram model with Negative Sampling. We store the trained weight matrices in `W₁_ns::Matrix{Float64}` and `W₂_ns::Matrix{Float64}`, and the per-epoch average loss in `loss_history_ns::Vector{Float64}`.

In [ ]:
W₁_ns, W₂_ns, loss_history_ns = let

    # hyperparameters -
    vocab_size = length(vocabulary);
    d_h = 5;           # embedding dimension
    η = 0.01;          # learning rate
    num_epochs = 500;
    k = 5;             # number of negative samples

    # initialize weight matrices -
    W₁ = randn(d_h, vocab_size) * 0.1;
    W₂ = randn(vocab_size, d_h) * 0.1;

    # sigmoid helper -
    σ(x) = 1.0 / (1.0 + exp(-x));

    # training loop -
    loss_history = zeros(num_epochs);
    sampler = Categorical(noise_distribution);
    for epoch ∈ 1:num_epochs
        epoch_loss = 0.0;
        for (x_target, y_context) in skipgram_pairs

            # get indices -
            target_idx = argmax(x_target);
            context_idx = argmax(y_context);

            # get embeddings -
            h = W₁[:, target_idx];                      # target embedding (column of W₁)
            w_pos = W₂[context_idx, :];                 # positive context embedding (row of W₂)

            # sample k negative words -
            neg_indices = [rand(sampler) for _ in 1:k];

            # forward pass: positive score -
            s_pos = dot(w_pos, h);
            loss = -log(σ(s_pos) + 1e-12);

            # gradient for positive example -
            ∂h = (σ(s_pos) - 1.0) * w_pos;
            W₂[context_idx, :] .-= η * (σ(s_pos) - 1.0) * h;

            # forward pass: negative scores -
            for ni in neg_indices
                w_neg = W₂[ni, :];
                s_neg = dot(w_neg, h);
                loss += -log(σ(-s_neg) + 1e-12);

                # gradient for negative example -
                ∂h .+= σ(s_neg) * w_neg;
                W₂[ni, :] .-= η * σ(s_neg) * h;
            end

            # update target embedding -
            W₁[:, target_idx] .-= η * ∂h;
            epoch_loss += loss;
        end
        loss_history[epoch] = epoch_loss / length(skipgram_pairs);
    end

    (W₁, W₂, loss_history)
end;
println("Negative Sampling training complete. Final loss: $(round(loss_history_ns[end], digits=4))")

Let's compare the training loss curves for both approaches.

In [ ]:
let
    plot(loss_history_softmax, label="Full Softmax", lw=2, xlabel="Epoch", ylabel="Average Loss")
    plot!(loss_history_ns, label="Negative Sampling (k=5)", lw=2, ls=:dash)
end

___

## Task 4: Compare Skip-Gram and CBOW Embeddings
To compare with CBOW, we train a CBOW model on the same corpus (replicating the L9a setup) and then compare cosine similarities for selected word pairs across all three models: CBOW, Skip-Gram (full softmax), and Skip-Gram (Negative Sampling).

> __Why compare?__
>
> CBOW and Skip-Gram optimize different objectives on the same data. CBOW predicts a target from context (averaging over context words), while Skip-Gram predicts each context word separately. We expect both to capture co-occurrence patterns, but Skip-Gram may produce different similarity rankings, particularly for less frequent words.

First, let's train a CBOW model and store the input weight matrix in the `W₁_cbow::Matrix{Float64}` variable.

In [ ]:
# Train CBOW model (same setup as L9a) -
W₁_cbow = let

    vocab_size = length(vocabulary);
    d_h = 5; η = 0.01; num_epochs = 500;
    W₁ = randn(d_h, vocab_size) * 0.1;
    W₂ = randn(vocab_size, d_h) * 0.1;

    # generate CBOW training pairs -
    window_size = 2;
    cbow_pairs = Array{Tuple{Vector{Float64}, Vector{Float64}}, 1}();
    for sentence in sentences
        augmented = "<bos> " * sentence * " <eos>";
        words = split(lowercase(augmented)) .|> String;
        idxs = [get(vocabulary, w, vocabulary["<unk>"]) for w in words];
        for i ∈ eachindex(idxs)
            target = zeros(Float64, vocab_size); target[idxs[i]] = 1.0;
            context = zeros(Float64, vocab_size);
            for j ∈ max(1,i-window_size):min(length(idxs),i+window_size)
                j == i && continue;
                context[idxs[j]] += 1.0;
            end
            push!(cbow_pairs, (context, target));
        end
    end

    # train -
    for epoch ∈ 1:num_epochs
        for (x, y) in cbow_pairs
            h = W₁ * x; u = W₂ * h; ŷ = softmax(u);
            δ = ŷ - y;
            W₂ .-= η * (δ * h');
            W₁ .-= η * ((W₂' * δ) * x');
        end
    end
    W₁
end;
println("CBOW training complete.")

Now let's compare cosine similarities across all three models.

In [ ]:
let
    function cosine_sim(w1::String, w2::String, W::Matrix{Float64}, vocab::Dict{String,Int64})
        v1 = W[:, vocab[w1]]; v2 = W[:, vocab[w2]];
        return dot(v1, v2) / (norm(v1) * norm(v2));
    end

    word_pairs = [
        ("machine", "learning"),
        ("data", "science"),
        ("fun", "great"),
        ("love", "enjoy"),
        ("machine", "?"),
    ];

    models = [
        ("CBOW", W₁_cbow),
        ("SG-Softmax", W₁_sg),
        ("SG-NegSamp", W₁_ns),
    ];

    # build results table -
    rows = [];
    for (w1, w2) in word_pairs
        row = Dict{String,Any}("Word Pair" => "$w1 / $w2");
        for (name, W) in models
            row[name] = round(cosine_sim(w1, w2, W, vocabulary), digits=4);
        end
        push!(rows, row);
    end

    df = DataFrame(rows);
    pretty_table(df, header=["Word Pair", "CBOW", "SG-Softmax", "SG-NegSamp"])
end

___

## Summary
In this example, we trained Skip-Gram models using both full softmax and Negative Sampling, and compared the resulting embeddings to CBOW.

> __Key Takeaways__:
> 
> * __Skip-Gram generates more training pairs__: Each target word produces one pair per context position, resulting in $2m$ times more training examples than CBOW for the same corpus. This reversal of input and output is the key structural difference.
> * __Negative Sampling replaces softmax__: Instead of normalizing over the full vocabulary, Negative Sampling uses sigmoid scores for the positive context word and $k$ random negative words. The loss curves show that both approaches converge, but Negative Sampling avoids the expensive softmax computation.
> * __Different models capture different patterns__: CBOW and Skip-Gram produce different similarity rankings for the same word pairs because they optimize different objectives. Skip-Gram tends to produce more distinctive embeddings for infrequent words due to its higher training signal per occurrence.

These results illustrate the trade-offs between CBOW and Skip-Gram, and between full softmax and Negative Sampling.
___